In [1]:
import argparse
import numpy as np
import os
from chunkchromatin.simulation import Simulation
from chunkchromatin.chromosome import Chromosome
from chunkchromatin.lamina import Lamina
from chunkchromatin.hdf5_format import HDF5Reporter
from chunkchromatin.simulation import EKExceedsError
from chunkchromatin.condensate import Condensate
import openmm as mm
from polykit.polykit.generators.initial_conformations import create_random_walk

import json

In [2]:
import numpy as np

def generate_block_copolymer(N, min_block_size=5, types=(0, 1, 2), seed=None):
    """
    Generate a block copolymer array of length N, with each block at least min_block_size long,
    and block types randomly alternating between the given types (no adjacent blocks have the same type).
    """
    if seed is not None:
        np.random.seed(seed)
    monomer_types = np.empty(N, dtype=int)
    current = 0
    last_type = None
    while current < N:
        # Choose a type different from the last block
        possible_types = [t for t in types if t != last_type]
        block_type = np.random.choice(possible_types)
        # Choose block length (at least min_block_size, but not to exceed N)
        max_block = N - current
        block_len = np.random.randint(min_block_size, max_block + 1) if max_block > min_block_size else max_block
        monomer_types[current:current+block_len] = block_type
        current += block_len
        last_type = block_type
    return monomer_types

def generate_condensate_positions(n_condensates, chromatin_positions, box_length, min_dist=1.0, seed=42):
    np.random.seed(seed)
    condensate_positions = []

    while len(condensate_positions) < n_condensates:
        trial_pos = np.random.uniform(0, box_length, size=3)
        dists = np.linalg.norm(chromatin_positions - trial_pos, axis=1)
        if np.all(dists > min_dist):
            condensate_positions.append(trial_pos)

    return np.array(condensate_positions)


In [7]:
# === System Setup ===
n_chromatin = 100
n_condensates = 10
N = n_chromatin + n_condensates
density = 0.33
box_length = (N / density) ** (1 / 3.)
chains = [(0, 50, False), (50, 100, False)]

# === Chromatin & Condensate Types ===
monomer_types = generate_block_copolymer(n_chromatin, min_block_size=5, types=(0, 1, 2), seed=42)
np.save("monomer_types.npy", monomer_types)

condensate_types = np.random.choice([0, 1], size=n_condensates, replace=True)
chromatin_types_full = np.concatenate([monomer_types, [-1] * n_condensates])

# === Interaction Matrices ===
interaction_matrix = np.array([
    [0.05, 0.05, 0.08],
    [0.05, 0.13, 0.17],
    [0.08, 0.17, 0.22]
])

epsilon_cc = np.array([
    [1.0, 0.8],
    [0.8, 1.2]
])

epsilon_cchr = np.array([
    [0.2, 0.3, 0.4],
    [0.1, 0.5, 0.6]
])

# === Reporter & Simulation Setup ===
out_dir = 'test_output_condensate'
os.makedirs(out_dir, exist_ok=True)
reporter = HDF5Reporter(folder=out_dir, max_data_length=500, overwrite=True)

sim = Simulation(
    integrator_type="variableLangevin",
    temperature=300.0,  # Kelvin
    gamma=0.05,         # 1/ps
    timestep=5,         # fs
    platform='CPU',
    N=N,
    reporter=reporter
)

# === Force Objects ===
chromosome = Chromosome(N, chains, sim)
lamina = Lamina(N, chains, sim)

# Add condensate–condensate and condensate–chromatin forces (added inside constructor)
condensate = Condensate(
    N=N,
    chains=chains,
    simulation=sim,
    condensate_types=condensate_types,
    chromatin_types=chromatin_types_full,
    epsilon_cc=epsilon_cc,
    epsilon_cchr=epsilon_cchr,
    cutoff=3.0,
    alpha=6
)

# === Initial Positions ===
chromatin_positions = create_random_walk(step_size=1, N=n_chromatin)
condensate_positions = generate_condensate_positions(
    n_condensates=n_condensates,
    chromatin_positions=chromatin_positions,
    box_length=box_length,
    min_dist=1.0
)

positions = np.vstack([chromatin_positions, condensate_positions])
sim.set_positions(positions)

# === Add Chromatin Forces ===
harmonic_bond_force = chromosome.add_harmonic_bond()
angle_force = chromosome.add_angle_force()

monomer_types_padded = np.concatenate([monomer_types, [-1] * n_condensates])
nonbonded_pair_potential_force = chromosome.add_nonbonded_pair_potential(
    sim,
    interaction_matrix,
    monomer_types_padded
)

spherical_confinement_force = lamina.add_spherical_confinement(sim)

sim.add_force(harmonic_bond_force)
sim.add_force(angle_force)
sim.add_force(nonbonded_pair_potential_force)
sim.add_force(spherical_confinement_force)

# === Finalize System ===
sim.create_context()
sim.set_velocities()
